In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval



In [7]:


# Login no TradingView
tv = TvDatafeed()

ticker = 'WIN1!'
exchange = 'BMFBOVESPA'


df = tv.get_hist(
    symbol=ticker,
    exchange=exchange,
    interval=Interval.in_daily,
    n_bars=10000
)
df = df[df.index.year >=2000].dropna()
df.index = pd.to_datetime(df.index).normalize().date
df.drop(columns='symbol',inplace=True)
df.index = pd.to_datetime(df.index).normalize()
df['weekday'] = df.index.weekday
df.dropna(inplace=True)
df['ret'] = df['close'].pct_change()
df.tail()




,open,high,low,close,volume,weekday,ret
2025-12-16,161250.0,161350.0,157325.0,158219.0,18346181.0,1,-0.028187
2025-12-17,161500.0,161690.0,159610.0,160617.0,17583079.0,2,0.015156
2025-12-18,160660.0,161835.0,159845.0,161296.0,17220599.0,3,0.004227
2025-12-19,161200.0,162725.0,160990.0,161574.0,14265906.0,4,0.001724
2025-12-22,162055.0,162415.0,160150.0,161083.0,15641312.0,0,-0.003039


In [8]:
roll_ret_win = 20
df['rooling_ret'] = (df['close'] / df['close'].shift(roll_ret_win)) - 1

df["position"] = np.where((df['rooling_ret'].shift(1) > df['rooling_ret']),
                        1,
                        0)


df["strategy_ret"] = (df["position"] * df["ret"].shift(-1))

# df.dropna(inplace=True)
df["buy_hold"] = df["ret"].add(1).cumprod().sub(1)
df["strategy"] = df["strategy_ret"].add(1).cumprod().sub(1)
df.tail()


,open,high,low,close,volume,weekday,ret,rooling_ret,position,strategy_ret,buy_hold,strategy
2025-12-16,161250.0,161350.0,157325.0,158219.0,18346181.0,1,-0.028187,-0.003106,1,0.015156,4.470920,6.516673
2025-12-17,161500.0,161690.0,159610.0,160617.0,17583079.0,2,0.015156,0.015002,0,0.000000,4.553838,6.516673
2025-12-18,160660.0,161835.0,159845.0,161296.0,17220599.0,3,0.004227,0.027618,0,0.000000,4.577317,6.516673
2025-12-19,161200.0,162725.0,160990.0,161574.0,14265906.0,4,0.001724,0.033392,0,-0.000000,4.586929,6.516673
2025-12-22,162055.0,162415.0,160150.0,161083.0,15641312.0,0,-0.003039,0.028581,1,NaN,4.569952,NaN


In [9]:
# =========================
# PLOT
# =========================
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}]]
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2)),
    secondary_y=False
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2)
    ),
    secondary_y=False
)

# Layout
fig.update_layout(
    title=f"{ticker} | Rolling Window",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    yaxis2_title="Position",
    legend=dict(x=0.01, y=0.99),
    template="plotly_white",
    height=600
)

# Ajuste do eixo secundário
fig.update_yaxes(range=[-0.05, 1.05], secondary_y=True)

fig.show()



In [10]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-2]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    df["strategy_ret"].mean() / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    df["ret"].mean() / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 457.00%
Strategy Return:   651.67%

Buy & Hold Vol: 26.62%
Strategy Vol:   19.25%

Buy & Hold Sharpe: 0.45
Strategy Sharpe:   0.61
